# LDA API

In [1]:
import json
import requests
import copy
import pandas as pd
from pathlib import Path
from my_secrets import secrets
from utility import lda_endpoint

Set up basic folder structure

In [2]:
# Base path
curr_dir = Path()

# LDA folders
lda_dir = curr_dir / 'LDA'
lda_input_dir = lda_dir / 'input'
lda_output_dir = lda_dir / 'output'

## Overview of root endpoint categories

In [9]:
r = requests.get(lda_endpoint('root')).content
lda_categories = json.loads(r)
for lda_cat, lda_cat_url in lda_categories.items():
    print(f"Requesting {lda_cat} at {lda_cat_url}...")
    try:
        lda_cat_content = requests.get(lda_cat_url).content
        lda_cat_content = json.loads(lda_cat_content)

        if type(lda_cat_content) == list: # Some constants returns as a list of dicts
            for _dict in lda_cat_content[:3]:
                print(f'{json.dumps(_dict, indent=4)}')
            continue

        for key, val in lda_cat_content.items():
            if type(val) == list:
                print(f'{key}: {json.dumps(val[0], indent=4)}')
            else:
                print(f'{key}: {val}')
    except Exception as e:
        print(f"Failed!\n{e}")

Requesting filings at https://lda.senate.gov/api/v1/filings/...
count: 1948605
next: https://lda.senate.gov/api/v1/filings/?page=2
previous: None
results: {
    "url": "https://lda.senate.gov/api/v1/filings/455edc06-55d1-41ed-878e-70a4040f953c/",
    "filing_uuid": "455edc06-55d1-41ed-878e-70a4040f953c",
    "filing_type": "MM",
    "filing_type_display": "Mid-Year Report",
    "filing_year": 1999,
    "filing_period": "mid_year",
    "filing_period_display": "Mid-Year (Jan 1 - Jun 30)",
    "filing_document_url": "https://lda.senate.gov/filings/public/filing/455edc06-55d1-41ed-878e-70a4040f953c/print/",
    "filing_document_content_type": "application/pdf",
    "income": null,
    "expenses": null,
    "expenses_method": null,
    "expenses_method_display": null,
    "posted_by_name": null,
    "dt_posted": "1905-06-24T00:00:00-05:00",
    "termination_date": null,
    "registrant_country": "United States of America",
    "registrant_ppb_country": "United States of America",
    "regi

## Filings

### Generate examples

In [6]:
url = lda_endpoint('filings')
filings_data = []
limit = 100
for year in range(2026, 2019, -1):

    if len(filings_data) >= limit:
        break

    print(year)
    page = 1
    while len(filings_data) < limit and url is not None:
        print("\t", f"page {page}")

        payload = {
            'filing_year': year,
            'ordering':'-dt_posted'
        }

        filings = requests.get(
            url,
            headers=secrets['LDA']['headers'],
            params=payload
        ).json()

        url = filings['next']

        filings_data += filings['results']

        page += 1
print(f"All {limit} records obtained")

2026
	 page 1
	 page 2
	 page 3
	 page 4
All 100 records obtained


In [8]:
lda_sample_input_path = lda_input_dir / f'lda_filings_sample_{limit}.json'
with open(lda_sample_input_path, mode='w') as f:
    f.write(json.dumps(filings_data, indent=4))

### Parse Examples

In [9]:
with open(lda_sample_input_path, mode='r') as f:
    filings_sample = json.loads(f.read())

# Creating lists to then turn into reference dataframes; ignoring lobbyists and government_entities for now
registrant_list = []
filing_client_list = []
lobbying_activity_list = []
# Passing by reference means the filings_sample list is updated by these changes,
# so a separate empty list doesn't need to be created


for filing in filings_sample:
    # Create a registrant lookup table
    registrant = filing['registrant']
    registrant_list.append(registrant)
    filing['registrant'] = registrant['id']

    # Create a filing client lookup table; client_id is its own key-value pair
    filing_client = filing['client']
    filing_client_list.append(filing_client)
    filing['client'] = filing_client['id']

    # Create a lobbying_activity lookup table
    lobbying_activities = filing['lobbying_activities']
    if lobbying_activities != []:
        for lobbying_activity in lobbying_activities:
            # Keep only lobbyists IDs as FK reference
            lobbyists = lobbying_activity['lobbyists']
            lobbyist_ids = [lobbyist['lobbyist']['id'] for lobbyist in lobbyists]
            lobbying_activity['lobbyist_ids'] = lobbyist_ids
            del lobbying_activity['lobbyists']

            # Keep only Gov entity IDs as FK reference
            gov_entities = lobbying_activity['government_entities']
            gov_entity_ids = [gov_entity['id'] for gov_entity in gov_entities]
            lobbying_activity['gov_entity_ids'] = gov_entity_ids
            del lobbying_activity['government_entities']

            # Assign FK reference back to filings table
            lobbying_activity['filing'] = filing['filing_uuid']

            # Append to list
            lobbying_activity_list.append(lobbying_activity)
    # No longer need an explicit reference in filing table
    del filing['lobbying_activities']

Save as dataframes and export to csv

In [105]:
# Convert to pandas dataframes
filings_sample = pd.DataFrame(filings_sample)
filing_client_sample = pd.DataFrame(filing_client_list)
lobbying_activity_sample = pd.DataFrame(lobbying_activity_list)
registrant_sample = pd.DataFrame(registrant_list)

# Save to .csv's
filings_sample.to_csv(lda_output_dir / f'lda_filings_sample_{limit}.csv')
filing_client_sample.to_csv(lda_output_dir / f'lda_filing_clients_sample_{limit}.csv')
lobbying_activity_sample.to_csv(lda_output_dir / f'lda_lobbying_activities_sample_{limit}.csv')
registrant_sample.to_csv(lda_output_dir / f'lda_registrant_sample_{limit}.csv')